# Lesson 5 : Foundry Tools

In Lesson 4, we have learned basic tool calling by using built-in hosted tools in Microsoft Foundry.  
Microsoft Foundry also provides various additional tools in the gallery (catalog), called Foundry tools - such as, SharePoint tool, Fabric data agent tool, OpenAPI-integrated tool, or 3rd-party tools.

In this exercise, we change MCP example in Lesson 4 to use "Microsoft Learn MCP server" tool.

> Note : Some Foundry tools will handle the identity.  
> When you need to provide the credential or identity (such as, OAuth access token) in tools, process authentication and authorization in your client (such as, showing the login UI, processing OAuth flow, etc) and pass the credential for tool's usage.  
> In Agent Framework, you can also process sign-in request from Foundry, but I won't go into the steps here. (See official documents or examples.)

## 1. Connect to Microsoft Learn in Foundry tools

Before writing code, please connect to "Microsoft Learn" tool in Microsoft Foundry as follows.

1. Open Foundry Portal.
2. Go to "Build" tab.
3. Select "Tools" menu.
4. Select "Microsoft Learn MCP server" in catalog, and establish connection.

After the connection is established, let's **copy the project connection id**.

## 2. Run with "Microsoft Learn" tool in Foundry

Now we create an agent with above.

First we create a ```FoundryChatClient``` object as usual.

In [1]:
from dotenv import load_dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity.aio import AzureCliCredential

load_dotenv()

credential = AzureCliCredential()
client = FoundryChatClient(credential=credential)

"Microsoft Learn MCP server" tool is one of MCP tools in Foundry.<br>
Please configure properties for MCP tool as follows - which is the same configuration in ```azure-ai-projects```.

**Please replace the following ```CONNECTED_RESOURCE_ID``` with your project connection id** (which is obtained above).

In [2]:
from agent_framework import Agent

# ToDo : fill your below settings
#       (e.g., /subscriptions/{AZURE_SUBSCRIPTION_ID}/resourceGroups/{RESOURCE_GROUP_NAME}/providers/Microsoft.CognitiveServices/accounts/{FOUNDRY_RESOURCE_NAME}/projects/{FOUNDRY_PROJECT_NAME}/connections/{CONNECTED_RESOURCE_NAME})
CONNECTED_RESOURCE_ID = "xxxxxxxxxx"

agent = Agent(
    name="MSTechKnowledgeAgentWithFoundry",
    client=client,
    instructions="You are an agent who answers technical questions about Microsoft products and services.",
    tools=[
        {
            "type": "mcp",
            "server_label": "mymcp01",
            "server_url": "https://learn.microsoft.com/api/mcp",
            "require_approval": "never",
            "project_connection_id": CONNECTED_RESOURCE_ID,
        }
    ],
)

Let's ask a technical question about Microsoft Azure, and verify that **the response includes the document reference**.

In [3]:
from IPython.display import Markdown, display

result = await agent.run("How to create an Azure storage account using Azure CLI ?")
display(Markdown(result.text))

To create an Azure Storage account with **Azure CLI**, do:

```bash
# Sign in (skip if using Azure Cloud Shell)
az login
```

1) **Create (or use) a resource group**
```bash
az group create \
  --name storage-rg \
  --location eastus
```

2) **Create a general-purpose v2 storage account**
> Storage account names must be **globally unique** (lowercase letters/numbers only, 3–24 chars).

```bash
az storage account create \
  --name <account-name> \
  --resource-group storage-rg \
  --location eastus \
  --sku Standard_LRS \
  --kind StorageV2 \
  --min-tls-version TLS1_2 \
  --allow-blob-public-access false
```

3) **Verify**
```bash
az storage account show \
  --name <account-name> \
  --resource-group storage-rg \
  --query "{name:name, location:primaryLocation, sku:sku.name, kind:kind, endpoints:primaryEndpoints}"
```

References:
- https://learn.microsoft.com/azure/storage/common/storage-account-create#create-a-storage-account
- https://learn.microsoft.com/cli/azure/storage/account?view=azure-cli-latest#az-storage-account-create

## Final note

All tools registered in your Foundry project has unique **project connection id**, and your agents built in Agent Framework can then connect to any tools in project by setting this id.

For available tool's type and configurations, see [API reference](https://learn.microsoft.com/en-us/python/api/azure-ai-projects/azure.ai.projects.models?view=azure-python-preview) in Azure AI Projects SDK.

> Note : Mostly 3rd party tools in Microsoft Foundry has MCP type.